In [5]:
import subprocess
import time
import itertools
import os
import pandas as pd
import glob
from datetime import datetime

# ================= 設定區 =================
dic = {
    'batch_size': [1000],
    'epoch': [1000],
    'layer': [4, 3, 2],
    'hidden': [32, 16],
    'data_version': [58],            # 確保版本號與訓練腳本一致
    'lr': [0.003, 0.001,  0.03, 0.01],
    'column': ['acceleration_X,acceleration_Y,acceleration_Z,gyroscope_X,gyroscope_Y,gyroscope_Z'],
    'folds': [1, 2, 3, 4, 5],
    # ==== DANN 新增網格搜尋參數 ====
    'lambda_dann': [0.5],            # 對抗損失強度，可依需求調整如 [0.5, 1.0, 2.0]
    'target_version': ['58/special_data'], # 自動對接 ../data_v58/special_data 結構
}

MAX_CONCURRENT_JOBS = 5  # 訓練與推論時的併發數
TRAIN_SCRIPT = "train.py" 
TEST_SCRIPT = "test.py"   
OUTPUT_LOG_DIR = "./test_log"
# =========================================

def get_combinations(params):
    keys = list(params.keys())
    values = list(params.values())
    for combo in itertools.product(*values):
        yield dict(zip(keys, combo))

def run_phase(phase_name, script_name, combinations, max_jobs):
    print(f"\n=== 開始執行階段: {phase_name} ===")
    total_jobs = len(combinations)
    running_processes = []
    
    for i, p in enumerate(combinations):
        # 💡 【關鍵修正點 1】: 將 --fold 參數名稱對接到 test.py 的 argparse
        cmd = [
            'python3', script_name,
            f'--batch_size={p["batch_size"]}',
            f'--epoch={p["epoch"]}',
            f'--layer={p["layer"]}',
            f'--hidden={p["hidden"]}',
            f'--data_version={p["data_version"]}',
            f'--lr={p["lr"]}',
            f'--column={p["column"]}',
            f'--fold={p["folds"]}', # 把本來的 --fold 改為 --fold
            f'--lambda_dann={p["lambda_dann"]}',
            f'--target_version={p["target_version"]}'
        ]
        
        # 💡 【關鍵修正點 2】: 移除 stderr=DEVNULL，允許把錯誤訊息印在螢幕上以利除錯
        proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=sys.stderr if 'sys' in globals() else None)
        running_processes.append(proc)
        
        # 進度顯示
        if i % 10 == 0:
            print(f"[{phase_name}] 進度: {i}/{total_jobs} (Running: {len(running_processes)})")

        # 控管併發數
        while len(running_processes) >= max_jobs:
            running_processes = [proc for proc in running_processes if proc.poll() is None]
            if len(running_processes) >= max_jobs:
                time.sleep(1)

    # 等待最後一批完成
    for proc in running_processes:
        proc.wait()
    print(f"=== {phase_name} 階段完成 ===\n")

def collect_results():
    print(f"=== 正在從 {OUTPUT_LOG_DIR} 彙整最新測試報告 ===")
    
    result_files = glob.glob(os.path.join(OUTPUT_LOG_DIR, "*_result.csv"))
    
    if not result_files:
        print(f"在 {OUTPUT_LOG_DIR} 找不到任何測試結果檔案。")
        return

    all_dfs = []
    for f in result_files:
        try:
            df = pd.read_csv(f)
            if not df.empty:
                all_dfs.append(df)
        except Exception as e:
            print(f"讀取 {f} 失敗: {e}")
    
    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        
        if "macro_f1" in final_df.columns:
            final_df = final_df.sort_values(by="macro_f1", ascending=False)
        elif "f1" in final_df.columns: 
            final_df = final_df.sort_values(by="f1", ascending=False)
            
        out_name = "Final_Report_DANN_v58.csv"
        final_df.to_csv(out_name, index=False)
        
        print("-" * 50)
        print(f"報告整合完成！共彙總 {len(all_dfs)} 筆測試數據。")
        print(f"最終報告已產出: {out_name}")
        print("-" * 50)
        print("Top 5 最佳模型表現 (已納入 mAP 跨域泛化指標):")
        
        try:
            print(final_df.head(5)[['run_name', 'acc', 'macro_f1', 'mAP', 'mAP_notTired_Tired', 'f1_notTired', 'f1_Tired']])
        except KeyError:
            print(final_df.head(5))
            
    else:
        print("沒有有效的數據可以合併。")

def main():
    import sys # 確保 sys 套件載入
    start_time = datetime.now()
    combinations = list(get_combinations(dic)) 
    
    # 階段 1: 訓練（如果需要重新跑訓練，請將下方這行的註解 # 拿掉）
    run_phase("Training", TRAIN_SCRIPT, combinations, max_jobs=MAX_CONCURRENT_JOBS)
    
    # 階段 2: 測試
    run_phase("Testing", TEST_SCRIPT, combinations, max_jobs=MAX_CONCURRENT_JOBS)
    
    # 階段 3: 彙整
    collect_results()

    end_time = datetime.now()
    print(f"總耗時: {end_time - start_time}")

if __name__ == "__main__":
    main()


=== 開始執行階段: Training ===
[Training] 進度: 0/120 (Running: 1)
[Training] 進度: 10/120 (Running: 5)
[Training] 進度: 20/120 (Running: 5)
[Training] 進度: 30/120 (Running: 5)
[Training] 進度: 40/120 (Running: 5)
[Training] 進度: 50/120 (Running: 5)
[Training] 進度: 60/120 (Running: 5)
[Training] 進度: 70/120 (Running: 5)
[Training] 進度: 80/120 (Running: 5)
[Training] 進度: 90/120 (Running: 5)
[Training] 進度: 100/120 (Running: 5)
[Training] 進度: 110/120 (Running: 5)
=== Training 階段完成 ===


=== 開始執行階段: Testing ===
[Testing] 進度: 0/120 (Running: 1)
[Testing] 進度: 10/120 (Running: 1)
[Testing] 進度: 20/120 (Running: 1)
[Testing] 進度: 30/120 (Running: 1)
[Testing] 進度: 40/120 (Running: 1)
[Testing] 進度: 50/120 (Running: 1)
[Testing] 進度: 60/120 (Running: 1)
[Testing] 進度: 70/120 (Running: 1)
[Testing] 進度: 80/120 (Running: 1)
[Testing] 進度: 90/120 (Running: 1)
[Testing] 進度: 100/120 (Running: 1)
[Testing] 進度: 110/120 (Running: 1)
=== Testing 階段完成 ===

=== 正在從 ./test_log 彙整最新測試報告 ===
------------------------------------------